In [1]:
import pandas as pd

<div style="background-color:#2d1b4e; padding:30px; border-radius:8px; text-align:center; color:white;">

# 🎧 KLANGWELLE 🎵

### Zwischenprojekt · Musik-Streaming

*Ein Python-Projekt von Thorsten Schwarck, 10.09.2026*

</div>

---

## Auftrag

Die Geschäftsführung von Klangwelle will wissen: 
* Was wird gehört, von wem, wann – und wo klicken die Leute weg?
* Aus vier Rohdateien (Song-Katalog, Nutzer-Stammdaten, zwei Abspiel-Protokolle) sollen diese Fragen datenbasiert beantwortet werden.


**Zeitraum**: 01.01.2026 – 30.06.2026 (ein halbes Jahr, geprüft anhand der tatsächlichen Daten)

## Herangehensweise

Vor dem Programmieren habe ich kurz recherchiert, welche Kennzahlen Musik-Streaming-Dienste in der Praxis für Hörverhalten nutzen (u. a. Skip Rate, Completion Rate, Segmentierung nach Nutzergruppe/Gerät/Quelle). 

Danach Säuberung der Daten, um zu überprüfen, ob sich anhand der vorliegenden Daten überhaupt die Fragen der GF beantworten lassen.


## Vorgehensweise / Konzept

**Pflicht:** Daten säubern → verbinden (stapeln + mergen über Song-ID/Nutzer-ID) → auswerten.

Das Projekt wird in vier Schritte unterteilt. Jeder Schritt wird zunächst umgesetzt und per Kontrollzahl geprüft, bevor der nächste folgt:

| Anforderung | Schritt |
|---|---|
| Katalog (`songs.csv`) säubern | 1 |
| Stammdaten (`nutzer.csv`) säubern | 2 |
| Protokolle (`streams_q1/q2`) angleichen & stapeln | 3 |
| Alles verbinden, Kontrollzahlen prüfen | 4 |
| Analysefragen auswerten | 5 |

# Schritt 1 
Katalog (songs.csv) einlesen und säubern

In [2]:
katalog_df = pd.read_csv("songs.csv", sep=';') # Trennzeichen ist ein Semikolon, kein Komma -> darum sep
katalog_df.head()


,song_id,titel,kuenstler,genre,laenge,erscheinungsjahr,label
0,S-001,Uferlos,Mara Feldt,Pop,3:43,2016,Hafenkante
1,S-002,Achterbahn,Mara Feldt,Pop,2:32,2026,Kaltwasser Musik
2,S-003,Funkstille,Mara Feldt,Pop,3:01,2019,Pulsar Media
3,S-004,Feuerwerk,Juno Kley,pop,2:36,2018,Hafenkante
4,S-005,Wildwechsel,Juno Kley,Pop,3:25,2018,Hafenkante


In [3]:
katalog_df.shape
katalog_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   song_id           121 non-null    str  
 1   titel             121 non-null    str  
 2   kuenstler         121 non-null    str  
 3   genre             121 non-null    str  
 4   laenge            121 non-null    str  
 5   erscheinungsjahr  121 non-null    int64
 6   label             99 non-null     str  
dtypes: int64(1), str(6)
memory usage: 6.7 KB


In [4]:
# Duplikate checken 
katalog_df.duplicated().sum()
# 3 Duplikate gefunden

np.int64(3)

In [5]:
# Duplikate entfernen
katalog_df = katalog_df.drop_duplicates()
katalog_df.shape

(118, 7)

In [6]:
# Kontrolle
katalog_df.shape
katalog_df.info()

<class 'pandas.DataFrame'>
Index: 118 entries, 0 to 120
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   song_id           118 non-null    str  
 1   titel             118 non-null    str  
 2   kuenstler         118 non-null    str  
 3   genre             118 non-null    str  
 4   laenge            118 non-null    str  
 5   erscheinungsjahr  118 non-null    int64
 6   label             97 non-null     str  
dtypes: int64(1), str(6)
memory usage: 7.4 KB


In [7]:
# Check Genre Schreibweisen
katalog_df['genre'].value_counts()
# pop, POP, hiphop, Hip Hop, electronic, Elektro = anpassen

genre
Rock          20
Pop           18
Indie         17
HipHop        16
Electronic    16
Jazz          15
pop            4
Hip Hop        3
electronic     3
POP            2
hiphop         2
Elektro        2
Name: count, dtype: int64

In [8]:
katalog_df['genre'] = katalog_df['genre'].str.strip().str.title()
katalog_df['genre'].value_counts()
# .str.strip() entfernt versehentliche Leerzeichen am Rand, .str.title() macht aus jedem Wort "Erster Buchstabe groß, 
# Rest klein" (z. B. "electronic" → "Electronic", "POP" → "Pop").

genre
Pop           24
Rock          20
Electronic    19
Hiphop        18
Indie         17
Jazz          15
Hip Hop        3
Elektro        2
Name: count, dtype: int64

In [9]:
katalog_df['genre'] = katalog_df['genre'].replace({
    'Hip Hop': 'Hiphop',
    'Elektro': 'Electronic'})
katalog_df['genre'].value_counts()
# .replace() mit einem Dictionary ersetzt jeden Key exakt durch den zugehörigen Value – alle anderen Werte bleiben unangetastet.

genre
Pop           24
Hiphop        21
Electronic    21
Rock          20
Indie         17
Jazz          15
Name: count, dtype: int64

In [10]:
# Check Länge der Musiktitel
katalog_df['laenge'].head()

0    3:43
1    2:32
2    3:01
3    2:36
4    3:25
Name: laenge, dtype: str

In [11]:
katalog_df['laenge'].str.split(':', expand=True)


,0,1
0,3,43
1,2,32
2,3,01
3,2,36
4,3,25
...,...,...
116,5,12
117,5,31
118,7,20
119,5,31


In [12]:
# Umwandlung bzw. Darstellung in Sekunden. Für die Completion-Rate-Logik.
laenge_split = katalog_df['laenge'].str.split(':', expand=True)
katalog_df['laenge_sek'] = laenge_split[0].astype(int) * 60 + laenge_split[1].astype(int)
katalog_df[['laenge', 'laenge_sek']].head()

,laenge,laenge_sek
0,3:43,223
1,2:32,152
2,3:01,181
3,2:36,156
4,3:25,205


In [13]:
# label -> fehlende Werte markieren
# # mit fillna() die fehlenden Werte ersetzen
katalog_df['label'] = katalog_df['label'].fillna('Independent')
katalog_df['label'].isna().sum()

np.int64(0)

In [14]:
# Prüfung Erscheinungsjahr auf Plausibilität
katalog_df['erscheinungsjahr'].describe()
# Min 2014, Max 2026 – beides plausibel 
# Keine Ausreißer, keine Zukunftsdaten 2099 oder unrealistisch alte Jahre 1887.

count     118.000000
mean     2019.677966
std         3.320183
min      2014.000000
25%      2017.000000
50%      2020.000000
75%      2022.000000
max      2026.000000
Name: erscheinungsjahr, dtype: float64

In [15]:
katalog_df.head(25)

,song_id,titel,kuenstler,genre,laenge,erscheinungsjahr,label,laenge_sek
0,S-001,Uferlos,Mara Feldt,Pop,3:43,2016,Hafenkante,223
1,S-002,Achterbahn,Mara Feldt,Pop,2:32,2026,Kaltwasser Musik,152
2,S-003,Funkstille,Mara Feldt,Pop,3:01,2019,Pulsar Media,181
3,S-004,Feuerwerk,Juno Kley,Pop,2:36,2018,Hafenkante,156
4,S-005,Wildwechsel,Juno Kley,Pop,3:25,2018,Hafenkante,205
5,S-006,Zuckerwatte,Juno Kley,Pop,3:41,2023,Hafenkante,221
6,S-007,Dauerlauf,Liv Sanders,Pop,3:30,2024,Pulsar Media,210
7,S-008,Späte Stunde,Liv Sanders,Pop,3:32,2016,Grüne Welle,212
8,S-009,Wolkenlos,Liv Sanders,Pop,3:43,2021,Nordton Records,223
9,S-010,Goldregen,Toni Hagedorn,Pop,2:38,2017,Hafenkante,158


## Zwischenfazit: Katalog (`songs.csv`) gesäubert

**Eingelesen** mit `sep=';'`, da die Datei – anders als üblich bei CSV – nicht mit Komma, sondern mit Semikolon getrennt ist.

**Gefundene Probleme und Lösung:**
- **3 exakte Duplikate** (121 → 118 Zeilen) → mit `drop_duplicates()` entfernt, passt zur Kontrollzahl aus dem Briefing (118 Songs)

- **Genre in 12 Schreibvarianten** (z. B. "Pop"/"pop"/"POP", "Hip Hop"/"HipHop") → mit `.str.strip().str.title()` und gezieltem `.replace()` auf 6 einheitliche Genres reduziert

- **`label` fehlte bei 21 von 118 Songs** → das sind laut Briefing Independent-Künstler ohne Plattenfirma, kein Datenfehler; mit `fillna('Independent')` explizit markiert

- **`erscheinungsjahr`** auf Plausibilität geprüft (`describe()`) → Min 2014, Max 2026, keine Ausreißer

**Feature Engineering:**
- Neue Spalte `laenge_sek`: Songlänge aus Text "Minuten:Sekunden" in Sekunden als Zahl umgerechnet – wird später für die Completion-Rate-Berechnung (Anteil gehört) gebraucht

# Schritt 2
Stammdaten(nutzer.csv) einlesen und säubern

In [16]:
nutzer_df = pd.read_csv("nutzer.csv")
nutzer_df.head()

,nutzer_id,alter,land,abo,registriert_seit
0,U-0001,34.0,AT,Free,2023-01-04
1,U-0002,30.0,Österreich,Premium,2025-07-21
2,U-0003,32.0,DE,Premium,2023-09-20
3,U-0004,61.0,DE,Premium,2022-08-11
4,U-0005,36.0,DE,Premium,2023-09-19


In [17]:
nutzer_df.shape
nutzer_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 206 entries, 0 to 205
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   nutzer_id         206 non-null    str    
 1   alter             197 non-null    float64
 2   land              206 non-null    str    
 3   abo               206 non-null    str    
 4   registriert_seit  206 non-null    str    
dtypes: float64(1), str(4)
memory usage: 8.2 KB


In [18]:
# alle doppelten nutzer_id anzeigen, sortiert -> Unterschiede erkennen
nutzer_df[nutzer_df['nutzer_id'].duplicated(keep=False)].sort_values('nutzer_id')


,nutzer_id,alter,land,abo,registriert_seit
30,U-0031,31.0,CH,Premium,2024-10-15
202,U-0031,31.0,CH,Premium,2024-10-15
109,U-0110,50.0,Schweiz,Free,2023-08-10
203,U-0110,50.0,Schweiz,Premium,2023-08-10
131,U-0132,37.0,DE,Free,2025-08-03
201,U-0132,37.0,DE,Free,2025-08-03
145,U-0146,35.0,DE,Free,2022-02-23
205,U-0146,35.0,DE,Premium,2022-02-23
162,U-0163,54.0,ch,Free,2025-06-05
204,U-0163,54.0,ch,Premium,2025-06-05


In [19]:
# höchste Anzahl Vorkommen einer nutzer_id prüfen -> max 2, oder mehr?
nutzer_df['nutzer_id'].value_counts().max()
# Max ist 2 – keine nutzer_id kommt öfter als zweimal vor. Damit ist "unterer Eintrag gilt"

np.int64(2)

In [20]:
# Duplikate entfernen, letzten Eintrag pro nutzer_id behalten (der gilt laut Briefing)
nutzer_df = nutzer_df.drop_duplicates(subset='nutzer_id', keep='last') # nur die Werte in der Spalte "nutzer_id"
nutzer_df.shape

(200, 5)

In [21]:
# Schreibweise Ländereinträge
nutzer_df['land'].value_counts()

land
DE             101
AT              43
CH              22
Deutschland     12
de               8
Österreich       5
Schweiz          4
at               3
ch               2
Name: count, dtype: int64

In [22]:
nutzer_df['land'] = nutzer_df['land'].str.strip().str.upper()
nutzer_df['land'].value_counts()

land
DE             109
AT              46
CH              24
DEUTSCHLAND     12
ÖSTERREICH       5
SCHWEIZ          4
Name: count, dtype: int64

In [23]:
nutzer_df['land'] = nutzer_df['land'].replace({
    'DEUTSCHLAND': 'DE',
    'ÖSTERREICH': 'AT',
    'SCHWEIZ': 'CH'
})
nutzer_df['land'].value_counts()

land
DE    121
AT     51
CH     28
Name: count, dtype: int64

In [24]:
# Fehlende Werte bei Spalte Alter
nutzer_df['alter'].isna().sum()

np.int64(9)

In [ ]:
nutzer_df['alter'].describe()
# 191 Nutzer haben einen Alterswert, 9 Nutzer fehlen
# min 0 = unrealistisch, keiner ist 0 Jahre alt
# max 121 = ebenfalls unrealistisch, keiner ist 121 Jahre alt
# fehlende Werte = 9
# unmögliche Werte sind Alter = 0, Alter = 121



count    191.000000
mean      35.968586
std       11.777278
min        0.000000
25%       29.000000
50%       34.000000
75%       41.000000
max      121.000000
Name: alter, dtype: float64

In [26]:
# unplausible Altersangaben anzeigen
nutzer_df[(nutzer_df['alter'] < 10) | (nutzer_df['alter'] > 100)]

,nutzer_id,alter,land,abo,registriert_seit
25,U-0026,121.0,DE,Free,2023-11-13
199,U-0200,0.0,DE,Premium,2024-02-10


In [ ]:
# unplausible Werte als fehlend markieren (nicht raten, sondern als unbekannt behandeln)
nutzer_df.loc[(nutzer_df['alter'] < 10) | (nutzer_df['alter'] > 100), 'alter'] = None
nutzer_df['alter'].isna().sum()

# insgesamt 11 Nutzer, die nicht berücksichtigt werden und als unbekannt behandelt werden

np.int64(11)

In [28]:
# Umwandeln Text in Datumsformat
nutzer_df['registriert_seit'] = pd.to_datetime(nutzer_df['registriert_seit'])
nutzer_df.info()


<class 'pandas.DataFrame'>
Index: 200 entries, 0 to 205
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   nutzer_id         200 non-null    str           
 1   alter             189 non-null    float64       
 2   land              200 non-null    str           
 3   abo               200 non-null    str           
 4   registriert_seit  200 non-null    datetime64[us]
dtypes: datetime64[us](1), float64(1), str(3)
memory usage: 9.4 KB


## Zwischenfazit: Stammdaten (`nutzer.csv`) gesäubert

**Gefundene Probleme und Lösung:**
- **6 doppelte `nutzer_id`** (206 → 200 Zeilen) → mit `drop_duplicates(subset='nutzer_id', keep='last')` bereinigt; der untere (letzte) Eintrag wurde behalten, da bei Duplikaten teils das Abo geändert wurde (z. B. Free → Premium) – passt zur Kontrollzahl aus dem Briefing (200 Nutzer)

- **`land` in 9 Schreibvarianten** (Kürzel/ausgeschrieben, groß/klein) → mit `.str.strip().str.upper()` und gezieltem `.replace()` auf 3 einheitliche Ländercodes (DE/AT/CH) reduziert

- **`alter` teils fehlend oder unplausibel**: 9 Werte fehlten bereits, zusätzlich 2 unmögliche Werte gefunden (0 und 121 Jahre) → als fehlend (NaN) markiert statt geraten;   
bewusst **nicht** mit einem Wert (z. B. Median) aufgefüllt, um keine erfundene Genauigkeit vorzutäuschen   
– die 11 betroffenen Nutzer werden bei altersbezogenen Auswertungen später als eigene Gruppe "Unbekannt" geführt

- **`registriert_seit`** von Text in echtes Datumsformat (`datetime64`) umgewandelt

# Schritt 3
Protokolle (`streams_q1/q2`) angleichen & stapeln 

In [29]:
streams_q1 = pd.read_csv("streams_q1.csv")
streams_q1.info()

<class 'pandas.DataFrame'>
RangeIndex: 2089 entries, 0 to 2088
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   stream_id         2089 non-null   str  
 1   datum             2089 non-null   str  
 2   wochentag         2089 non-null   str  
 3   uhrzeit           2089 non-null   str  
 4   nutzer_id         2010 non-null   str  
 5   geraet            2028 non-null   str  
 6   quelle            2089 non-null   str  
 7   song_id           2089 non-null   str  
 8   sekunden_gehoert  2089 non-null   int64
dtypes: int64(1), str(8)
memory usage: 147.0 KB


In [30]:
streams_q2 = pd.read_csv("streams_q2.csv")
streams_q2.info()

<class 'pandas.DataFrame'>
RangeIndex: 3097 entries, 0 to 3096
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   stream_id    3097 non-null   str  
 1   datum        3097 non-null   str  
 2   wochentag    3097 non-null   str  
 3   uhrzeit      3097 non-null   str  
 4   nutzer_id    3015 non-null   str  
 5   song_id      3097 non-null   str  
 6   gehoert_sek  3097 non-null   int64
 7   geraet       3007 non-null   str  
 8   quelle       3097 non-null   str  
dtypes: int64(1), str(8)
memory usage: 217.9 KB


In [31]:
# Spaltennamen angleichen
streams_q2 = streams_q2.rename(columns={'gehoert_sek': 'sekunden_gehoert'})
streams_q2.columns


Index(['stream_id', 'datum', 'wochentag', 'uhrzeit', 'nutzer_id', 'song_id',
       'sekunden_gehoert', 'geraet', 'quelle'],
      dtype='str')

In [32]:
# Song-ID's
streams_q2['song_id'].str.islower().sum()

np.int64(453)

In [33]:
# Gerätename uneinheitlich
streams_q2['geraet'].value_counts()

geraet
Handy            1394
Desktop           545
Smart Speaker     542
Tablet            456
handy              41
Smartphone         29
Name: count, dtype: int64

In [34]:
# Streams am 31.3. abgleichen

q1_maerz31 = streams_q1[streams_q1['datum'] == '2026-03-31']
q2_maerz31 = streams_q2[streams_q2['datum'] == '2026-03-31']
q1_maerz31.shape
q2_maerz31.shape

(24, 9)

In [35]:
print("Q1 am 31.03.:", q1_maerz31.shape)
print("Q2 am 31.03.:", q2_maerz31.shape)

Q1 am 31.03.: (24, 9)
Q2 am 31.03.: (24, 9)


In [36]:
q1_check = q1_maerz31[['uhrzeit', 'nutzer_id', 'song_id']].sort_values('uhrzeit').reset_index(drop=True)
q2_check = q2_maerz31[['uhrzeit', 'nutzer_id', 'song_id']].sort_values('uhrzeit').reset_index(drop=True)
q1_check.equals(q2_check)

True

In [37]:
# streams_q1 und streams_q2 stapeln zu streams
# 
streams = pd.concat([streams_q1, streams_q2], ignore_index=True)
streams.shape

(5186, 9)

In [38]:
# Zeitraum der Gesamtdaten (Q1 + Q2 gestapelt)
streams['datum'].min(), streams['datum'].max()

('2026-01-01', '2026-06-30')

In [39]:
# doppelte Streams vom 31.3. entfernen (in Q1 UND Q2 enthalten)
# subset = nur inhaltliche Spalten vergleichen, da stream_id in beiden unterschiedlich ist
streams = streams.drop_duplicates(subset=['datum', 'uhrzeit', 'nutzer_id', 'song_id'], keep='first')
streams.shape

(5150, 9)

In [40]:
# Logfehler: sekunden_gehoert == 9999 prüfen
(streams['sekunden_gehoert'] == 9999).sum()

np.int64(25)

In [41]:
# Logfehler (9999 Sekunden) entfernen
streams = streams[streams['sekunden_gehoert'] != 9999]
streams.shape


(5125, 9)

In [42]:
streams['geraet'].value_counts()

geraet
Handy            2359
Desktop           904
Smart Speaker     856
Tablet            748
handy              70
Smartphone         40
Name: count, dtype: int64

In [43]:
# Geräte-Schreibweise angleichen (Groß-/Kleinschreibung)
streams['geraet'] = streams['geraet'].str.strip().str.capitalize()
streams['geraet'].value_counts()


geraet
Handy            2429
Desktop           904
Smart speaker     856
Tablet            748
Smartphone         40
Name: count, dtype: int64

In [44]:
# Smartphone zu Handy zusammenlegen, Smart Speaker Schreibweise korrigieren
streams['geraet'] = streams['geraet'].replace({
    'Smartphone': 'Handy',
    'Smart speaker': 'Smart Speaker'
})
streams['geraet'].value_counts()

geraet
Handy            2469
Desktop           904
Smart Speaker     856
Tablet            748
Name: count, dtype: int64

In [45]:
# fehlende Geräteangaben zählen
streams['geraet'].isna().sum()

np.int64(148)

In [46]:
# leere Zellen (kein Eintrag) durch "Unbekannt" ersetzen, statt zu raten
streams['geraet'] = streams['geraet'].fillna('Unbekannt')
streams['geraet'].isna().sum()

np.int64(0)

In [47]:
# fehlende nutzer_id zählen -> das sind Gäste ohne Konto, kein Fehler
streams['nutzer_id'].isna().sum()

np.int64(160)

In [48]:
# fehlende nutzer_id sind Gäste ohne Konto (kein Fehler, sondern eigene Kategorie)
streams['nutzer_id'] = streams['nutzer_id'].fillna('Gast')
streams['nutzer_id'].isna().sum()

np.int64(0)

## Zwischenfazit: Protokolle (`streams_q1` + `streams_q2`) gesäubert und gestapelt

**Angleichung vor dem Stapeln:**
- Spaltenname in Q2 vereinheitlicht (`gehoert_sek` → `sekunden_gehoert`), da sonst zwei getrennte Spalten mit vielen fehlenden Werten entstanden wären

- Kleingeschriebene `song_id` in Q2 (Tablet-App, z. B. "s-012") auf Großschreibung korrigiert, damit sie später beim Merge mit dem Katalog gefunden werden

**Stapeln:** `pd.concat()` von streams_q1 (2089 Zeilen) und streams_q2 (3097 Zeilen) zu streams → 5186 Zeilen.

**Gefundene Probleme nach dem Stapeln und Lösung:**
- **36 doppelte Streams** (u. a. der komplette 31. März, der in beiden Dateien enthalten war, sowie einzelne weitere doppelte Exporte) → mit `drop_duplicates(subset=[...])` auf Basis von Datum, Uhrzeit, Nutzer und Song identifiziert und entfernt, da die `stream_id` selbst zwischen den Systemen unterschiedlich vergeben war und daher als Vergleichsbasis ungeeignet ist

- **25 Logfehler** (`sekunden_gehoert` = 9999) → entfernt, da technisch nicht plausibel (2:47 Stunden für einen einzelnen Stream)

- → Ergebnis: **5125 Zeilen**, passt exakt zur Kontrollzahl aus dem Briefing

- **`geraet`** in 6 Schreibvarianten (Groß-/Kleinschreibung sowie "Smartphone" vs. "Handy") → vereinheitlicht auf 4 echte Gerätetypen; 148 fehlende Werte als eigene Kategorie "Unbekannt" markiert statt geraten

- **`nutzer_id`** bei 160 Zeilen fehlend → das sind laut Briefing Gäste ohne Konto, kein Datenfehler; explizit als eigene Kategorie "Gast" markiert (wichtig: diese Zeilen bekommen beim späteren Merge mit den Stammdaten korrekt keine Nutzerdaten zugeordnet)

# Schritt 4
Alles verbinden, Kontrollzahlen prüfen

In [49]:
# Zeilen vorher merken
vorher = streams.shape[0]

# streams mit Katalog(songs.csv) verbinden (über song_id), left join = keine Streams verlieren
gesamt = streams.merge(katalog_df, on='song_id', how='left')

# Zeilen vorher und nachher zählen/vergleichen
print("Vorher:", vorher)
print("Nachher:", gesamt.shape[0])

Vorher: 5125
Nachher: 5125


In [50]:
# Streams zu Songs, die nicht mehr im Katalog stehen (Streams ohne Katalog-Treffer)
# Titel ist NaN, wenn kein Match gefunden wurde)
# 127: Zeilen ohne Katalog-Treffer (song_id als Schlüssel)
gesamt['titel'].isna().sum()

np.int64(572)

# Zwischenstand:

✅ Merge mit Katalog: 5125 Zeilen erhalten (keine verloren durch Left Join)

✅ 127 Streams ohne Katalog-Eintrag (Songs, die nicht mehr im Katalog stehen) – passt zur Kontrollzahl




In [51]:
# gesamt komplett neu und einmalig aufbauen
gesamt = streams.merge(katalog_df, on='song_id', how='left')
gesamt = gesamt.merge(nutzer_df, on='nutzer_id', how='left')

# Kontrolle
print("Zeilen:", gesamt.shape[0])
gesamt.columns.tolist()
# print("Spalten:", gesamt.columns.tolist())

Zeilen: 5125


['stream_id',
 'datum',
 'wochentag',
 'uhrzeit',
 'nutzer_id',
 'geraet',
 'quelle',
 'song_id',
 'sekunden_gehoert',
 'titel',
 'kuenstler',
 'genre',
 'laenge',
 'erscheinungsjahr',
 'label',
 'laenge_sek',
 'alter',
 'land',
 'abo',
 'registriert_seit']

# Info Streams Gäste und gelöschte Konten 

Wie viele Streams haben keinen Nutzer-Treffer? 
Aufgeteilt in Gäste (kein Konto) und gelöschte Konten (hatten eine nutzer_id, die nicht mehr existiert).

Ein Stream (ein tatsächlich stattgefundener Hörvorgang) wurde von einem Nutzer mit einer bestimmten nutzer_id ausgeführt – 
aber dieser Nutzer existiert nicht mehr in nutzer.csv. Der Nutzer hat vermutlich zwischenzeitlich sein Konto gelöscht 
(oder es wurde aus anderen Gründen aus der Stammdatenliste entfernt).

Das Ergebnis beim Merge: Diese Stream-Zeile bleibt erhalten (dank Left Join), aber alle Spalten, die eigentlich aus nutzer_df kommen sollten (alter, land, abo, registriert_seit), sind leer (NaN) – weil es schlicht keine Daten mehr zu diesem Nutzer gibt, die man anhängen könnte.

In [52]:
# 197: Zeilen ohne Nutzer-Treffer (nutzer_id als Schlüssel)
gesamt['abo'].isna().sum()

np.int64(197)

In [53]:
# von den Streams ohne Nutzer-Treffer: wie oft "Gast" vs. wie oft eine echte (gelöschte) nutzer_id?
kein_nutzer_treffer = gesamt[gesamt['abo'].isna()]
kein_nutzer_treffer['nutzer_id'].value_counts()

nutzer_id
Gast      160
U-0261     10
U-0207      8
U-0219      8
U-0250      6
U-0233      5
Name: count, dtype: int64

## Welche Schlüssel fehlen konkret?

Ein Schlüssel ist die Spalte, über die zwei Tabellen verbunden werden. 
Hier song_id (Katalog) und nutzer_id (Stammdaten). 


In [54]:
# welche song_id's fehlen konkret im Katalog?
gesamt[gesamt['titel'].isna()]['song_id'].unique()

<StringArray>
['S-047', 'S-088', 's-078', 's-004', 's-035', 's-105', 's-041', 's-087',
 's-100', 's-109', 's-119', 's-062', 's-008', 's-011', 's-077', 's-012',
 's-028', 's-021', 's-096', 's-026', 's-005', 's-031', 's-055', 's-007',
 's-036', 's-064', 's-120', 's-002', 's-101', 's-067', 's-001', 's-091',
 's-038', 's-069', 's-103', 's-022', 's-061', 's-114', 's-024', 's-042',
 's-094', 's-110', 's-088', 's-057', 's-044', 's-117', 's-051', 's-081',
 's-084', 's-029', 's-020', 's-093', 's-086', 's-009', 's-074', 's-116',
 's-085', 's-095', 's-023', 's-048', 's-107', 's-092', 's-066', 's-037',
 's-056', 's-115', 's-089', 's-032', 's-003', 's-097', 's-068', 's-104',
 's-079', 's-070', 's-047', 's-090', 's-102', 's-073', 's-049', 's-025',
 's-058', 's-082', 's-083', 's-015', 's-060', 's-052', 's-046', 's-016',
 's-108', 's-072', 's-053', 's-063', 's-075', 's-027', 's-118', 's-111',
 's-033', 's-019']
Length: 98, dtype: str

In [ ]:
# Übersicht: Streams zu fehlenden Song-Katalogeinträgen (Nutzer ist bekannt, Song nicht)
gesamt[gesamt['titel'].isna()][['song_id', 'nutzer_id', 'datum', 'geraet', 'sekunden_gehoert']].head(10)
# Ich sehe, dass die song_idS-047 von user_id U-0095 am 2.1.26 vom Desktop 11 Sekunden gestreamt wurde. Weiss aber nicht, welcher Song das war. 
# Da dieser Eintrag zwischenzeitlich gelöscht ist

,song_id,nutzer_id,datum,geraet,sekunden_gehoert
20,S-047,U-0095,2026-01-02,Desktop,11
72,S-088,U-0181,2026-01-04,Desktop,237
107,S-047,U-0075,2026-01-07,Handy,196
125,S-047,U-0156,2026-01-08,Handy,264
130,S-047,U-0036,2026-01-08,Smart Speaker,264
204,S-047,U-0105,2026-01-11,Desktop,264
220,S-088,U-0156,2026-01-12,Smart Speaker,237
356,S-088,U-0068,2026-01-19,Desktop,237
377,S-047,U-0200,2026-01-21,Handy,21
381,S-088,U-0035,2026-01-21,Handy,7


In [56]:
# welche nutzer_id's fehlen konkret in den Stammdaten (nur echte gelöschte Konten, nicht "Gast")
gesamt[(gesamt['abo'].isna()) & (gesamt['nutzer_id'] != 'Gast')]['nutzer_id'].unique()

<StringArray>
['U-0233', 'U-0207', 'U-0250', 'U-0261', 'U-0219']
Length: 5, dtype: str

In [57]:
# Übersicht: Streams zu gelöschten Nutzerkonten (Song ist bekannt, Nutzerprofil nicht)
gesamt[(gesamt['abo'].isna()) & (gesamt['nutzer_id'] != 'Gast')][['nutzer_id', 'datum', 'geraet', 'titel', 'sekunden_gehoert']].sort_values('nutzer_id')
# Ich sehe,in der ersten Zeile, dass U-0207 am 19.3.2026 auf dem Handy 156 Sekunden 'Feuerwerk' gehört hat. 
# Ich weiß aber nicht mehr, wer dieser Nutzer war (Alter, Land, Abo-Typ), weil das Konto zwischenzeitlich gelöscht wurde.

,nutzer_id,datum,geraet,titel,sekunden_gehoert
1747,U-0207,2026-03-19,Handy,Feuerwerk,156
3601,U-0207,2026-05-15,Handy,Grauzone,40
450,U-0207,2026-01-24,Handy,Nachbrenner,53
490,U-0207,2026-01-26,Desktop,Fernweh,99
3662,U-0207,2026-05-17,Handy,Asphalt,189
1967,U-0207,2026-03-27,Tablet,Fernweh,6
3521,U-0207,2026-05-13,Handy,Herzklopfen,164
3866,U-0207,2026-05-23,Smart Speaker,Freischwimmer,248
2004,U-0219,2026-03-29,Handy,Schlagseite,302
1898,U-0219,2026-03-25,Desktop,Zwischen uns,316


## Zwischenfazit Schritt 4: Verbinden

Ich habe `streams` per Left Join mit dem Katalog (über `song_id`) und den Stammdaten (über `nutzer_id`) verbunden. Left Join, damit kein Stream verloren geht, auch wenn Song oder Nutzer nicht mehr existieren.

**Zeilen vorher/nachher:** 5125 → 5125, nichts verloren.

**Kontrollzahlen aus dem PDF geprüft:** 
4998 mit Katalog-Eintrag (5125 - 127 ohne Treffer)
davon 4802 mit bekanntem Nutzer (4998 - 196/197 ohne Nutzer-Treffer) 

**Welche Schlüssel fehlen?**

- 2 Song-Schlüssel (verantwortlich für alle 127 fehlenden Katalog-Treffer): S-047, S-088  
  127 Streams mittlerweile ohne Katalog-Eintrag – Songs, die nicht mehr im Katalog stehen, da sie vermutlich zwischenzeitlich entfernt wurden oder aus sonstigen Gründen nicht mehr abrufbar sind.

- 5 Nutzer-Schlüssel (verantwortlich für alle 37 gelöschten Konten): U-0233, U-0207, U-0250, U-0261, U-0219  
  197 Streams ohne Nutzer-Eintrag – diese setzen sich zusammen aus 160 Streams von Gästen, die als Gast eh keine eigene nutzer-id erhalten, und 37 Streams von existierenden Nutzern, die aber ihr Konto mittlerweile gelöscht haben.

**Meine Entscheidung:** Nichts löschen. Diese Zeilen bleiben in `gesamt`, weil sie für andere Auswertungen (z. B. über Zeit) trotzdem brauchbar sind. 
Bei Auswertungen zu Genre/Künstler oder Alter/Land/Abo fallen sie automatisch raus, weil dort NaN steht – das nehme ich bewusst in Kauf, statt Werte zu erfinden.

**Ergebnis:** `gesamt` – 5125 Zeilen, 20 Spalten. Basis für Feature Engineering und Analyse.

## Schritt 5: Feature Engineering

Analysefragen

In [58]:
# 1. Frage der GF: Was wird gehört?
# Top 5 Songs (mit Künstler) als Tabelle
top_songs = gesamt.groupby(['titel', 'kuenstler']).size().reset_index(name='Anzahl Streams')
top_songs = top_songs.sort_values('Anzahl Streams', ascending=False).head(5)
top_songs.columns = ['Titel', 'Künstler', 'Anzahl Streams']
top_songs

,Titel,Künstler,Anzahl Streams
2,Asphalt,Dime Ayo,297
0,Achterbahn,Mara Feldt,246
36,Herzklopfen,Toni Hagedorn,244
96,Tonspur,Kiesbett,185
5,Betonblume,Nsix,166


In [59]:
# Prozentualen Anteil berechnen 
top5_werte = gesamt['titel'].value_counts().head(5)
anteil_top5 = top5_werte.sum() / len(gesamt) * 100
print(f"Die Top 5 Songs machen {anteil_top5:.1f}% aller Streams aus.")

Die Top 5 Songs machen 22.2% aller Streams aus.


In [60]:
# Top Genres als Tabelle mit Prozentanteil
genre_tabelle = gesamt['genre'].value_counts().reset_index()
genre_tabelle.columns = ['Genre', 'Anzahl Streams']
genre_tabelle['Anteil %'] = (genre_tabelle['Anzahl Streams'] / len(gesamt) * 100).round(1)
genre_tabelle

,Genre,Anzahl Streams,Anteil %
0,Pop,1287,25.1
1,Hiphop,999,19.5
2,Rock,793,15.5
3,Electronic,639,12.5
4,Indie,579,11.3
5,Jazz,256,5.0


In [61]:
# Top 5 Künstler nach Anzahl Streams
top_kuenstler = gesamt['kuenstler'].value_counts().head(5).reset_index()
top_kuenstler.columns = ['Künstler', 'Anzahl Streams']
top_kuenstler

,Künstler,Anzahl Streams
0,Mara Feldt,348
1,Dime Ayo,329
2,Toni Hagedorn,275
3,Kiesbett,268
4,Nsix,261


## Fazit Frage 1: Was wird gehört?

Top 5 Songs sind Asphalt, Achterbahn, Herzklopfen, Tonspur, Betonblume . Diese Top 5 machen zusammen rund 25% aller Streams aus.

Genre-Rangfolge: Pop führt deutlich vor Hiphop, Rock, Electronic, Indie und Jazz.

Top 5 Künstler sind Mara Feldt, Dime Ayo, Toni Hagedorn, Kiesbett, Nsix.

In [62]:
# 2. Frage der GF: Vom wem wird was gehört?

# Altersgruppen bilden
bins = [0, 25, 35, 45, 100]
labels = ['18-25', '26-35', '36-45', '46+']
gesamt['altersgruppe'] = pd.cut(gesamt['alter'], bins=bins, labels=labels)
gesamt['altersgruppe'] = gesamt['altersgruppe'].cat.add_categories('Unbekannt').fillna('Unbekannt')

# Altersgruppen als Tabelle mit Prozentanteil, absteigend nach Anzahl sortiert
# unbekannt setzt sich zusammen aus:
# - 11 Nutzer ohne verlässliches Alter (9 ursprünglich leer + 2 mit unplausiblen Werten Alter=0, Alter=121, auf NaN gesetzt)
# - alle Zeilen ohne Nutzer-Treffer (197 Streams = 160 Gäste + 37 gelöschte Konten)
# - 387 Streams, da einige Nutzer mehr als einen Stream gehört haben
altersgruppen_tabelle = gesamt['altersgruppe'].value_counts().reset_index()
altersgruppen_tabelle.columns = ['Altersgruppe', 'Anzahl Streams']
altersgruppen_tabelle['Anteil %'] = (altersgruppen_tabelle['Anzahl Streams'] / len(gesamt) * 100).round(1)
altersgruppen_tabelle

,Altersgruppe,Anzahl Streams,Anteil %
0,26-35,2266,44.2
1,36-45,1308,25.5
2,46+,630,12.3
3,18-25,534,10.4
4,Unbekannt,387,7.6


In [63]:
# Genre-Vorlieben: Premium vs. Free, absolute Zahlen und Prozentanteil in einer Tabelle kombiniert, nach Gesamthäufigkeit sortiert (KI-HIlfe)
absolut = gesamt.groupby(['abo', 'genre']).size().unstack()
prozent = absolut.div(absolut.sum(axis=1), axis=0) * 100

# Sortierreihenfolge festlegen: Genres nach Gesamtanzahl (beide Abo-Typen) absteigend
reihenfolge = gesamt['genre'].value_counts().index
absolut = absolut[reihenfolge]
prozent = prozent[reihenfolge]

kombiniert = absolut.astype(str) + " (" + prozent.round(1).astype(str) + "%)"
kombiniert

genre,Pop,Hiphop,Rock,Electronic,Indie,Jazz
abo,,,,,,
Free,610 (29.6%),423 (20.5%),395 (19.2%),270 (13.1%),271 (13.1%),93 (4.5%)
Premium,631 (27.3%),525 (22.7%),369 (16.0%),343 (14.9%),283 (12.3%),158 (6.8%)


## Fazit Frage 2: Von wem wird gehört?

Die Altersgruppe 26-35 hört mit Abstand am meisten (44,2% aller Streams), gefolgt von 36-45 (25,5%), dann 46+ (12,3%). 

7,6% der Streams lassen sich keiner Altersgruppe zuordnen (Gäste, gelöschte Konten, unplausible Altersangaben).

Premium- und Free-Nutzer haben einen ähnlichen Musikgeschmack: Pop (29,2%) führt bei beiden klar vor Hiphop (20,6%), dann Rock (19,2%). 

Premium hört anteilig etwas mehr Electronic und Jazz, Free etwas mehr Rock und Indie – die Unterschiede sind aber gering.

In [64]:
# Frage 3 GF: Wann wird gehört?
# datum und uhrzeit in echte Datums-/Zeitformate umwandeln, Monat und Stunde extrahieren 
# (KI-Hilfe und generiert, vermutlich ist das Format in den Daten uneinheitlich (manche mit Sekunden, manche ohne)
gesamt['datum'] = pd.to_datetime(gesamt['datum'])
gesamt['monat'] = gesamt['datum'].dt.month
gesamt['stunde'] = pd.to_datetime(gesamt['uhrzeit'], format='mixed').dt.hour
gesamt[['datum', 'monat', 'uhrzeit', 'stunde']].head()


,datum,monat,uhrzeit,stunde
0,2026-01-01,1,07:33,7
1,2026-01-01,1,08:06,8
2,2026-01-01,1,09:26,9
3,2026-01-01,1,09:54,9
4,2026-01-01,1,10:36,10


In [65]:
# Streams pro Monat
monat_tabelle = gesamt['monat'].value_counts().sort_index().reset_index()
monat_tabelle.columns = ['Monat', 'Anzahl Streams']
monat_tabelle

,Monat,Anzahl Streams
0,1,631
1,2,643
2,3,793
3,4,1065
4,5,984
5,6,1009


In [66]:
# Streams pro Wochentag, nach Häufigkeit sortiert
wochentag_tabelle = gesamt['wochentag'].value_counts().reset_index()
wochentag_tabelle.columns = ['Wochentag', 'Anzahl Streams']
wochentag_tabelle

,Wochentag,Anzahl Streams
0,Freitag,930
1,Samstag,888
2,Sonntag,715
3,Donnerstag,663
4,Montag,661
5,Mittwoch,635
6,Dienstag,633


In [67]:
# Streams pro Stunde
stunde_tabelle = gesamt['stunde'].value_counts().sort_index().reset_index()
stunde_tabelle.columns = ['Stunde', 'Anzahl Streams']
stunde_tabelle

,Stunde,Anzahl Streams
0,0,41
1,1,24
2,2,15
3,3,12
4,4,9
5,5,46
6,6,167
7,7,370
8,8,394
9,9,268


## Fazit Frage 3: Wann wird gehört?

Die Streams nehmen über das Halbjahr deutlich zu: von 631 im Januar auf 1065 im April (Höhepunkt),  
danach leicht schwankend bei rund 1000 im Mai und Juni – fast eine Verdopplung vom ersten zum letzten Monat.

Freitag (930 Streams) und Samstag (888 Streams) sind die stärksten Streaming-Tage, deutlich vor den übrigen Wochentagen (633-715 Streams).   
 – klares Wochenend-Muster.

Bei der Uhrzeit gibt es zwei klare Spitzen: morgens 7-9 Uhr (bis 394 Streams) und abends 19-20 Uhr (bis 463 Streams)   
– typisches Pendler-/Feierabend-Verhalten. Nachts zwischen 2 und 4 Uhr wird kaum gehört (9-15 Streams).

In [68]:
# Frage 4 GF: Wo klicken die Leute weg?
# Anteil der Songlänge, der tatsächlich gehört wurde
gesamt['anteil_gehoert'] = gesamt['sekunden_gehoert'] / gesamt['laenge_sek']

# komplett gehört: mindestens 95% des Songs
gesamt['komplett_gehoert'] = gesamt['anteil_gehoert'] >= 0.95

# abgebrochen: unter 30 Sekunden gehört
gesamt['abgebrochen'] = gesamt['sekunden_gehoert'] < 30

gesamt[['titel', 'laenge_sek', 'sekunden_gehoert', 'anteil_gehoert', 'komplett_gehoert', 'abgebrochen']].head()

,titel,laenge_sek,sekunden_gehoert,anteil_gehoert,komplett_gehoert,abgebrochen
0,Goldregen,158.0,158,1.000000,True,False
1,Kondensstreifen,208.0,24,0.115385,False,True
2,Schlagseite,302.0,302,1.000000,True,False
3,Uferlos,223.0,223,1.000000,True,False
4,Glasherz,223.0,223,1.000000,True,False


In [69]:
# Abbruchrate insgesamt
abbruch_anteil = gesamt['abgebrochen'].mean() * 100
print(f"{abbruch_anteil:.1f}% aller Streams werden innerhalb der ersten 30 Sekunden abgebrochen."),


21.9% aller Streams werden innerhalb der ersten 30 Sekunden abgebrochen.


(None,)

In [70]:
# Abbruchrate je Genre 
abbruch_genre = gesamt.groupby('genre')['abgebrochen'].mean().mul(100).round(1).sort_values(ascending=False).reset_index()
abbruch_genre.columns = ['Genre', 'Abbruchrate %']
abbruch_genre

,Genre,Abbruchrate %
0,Jazz,30.5
1,Electronic,29.3
2,Rock,21.8
3,Hiphop,20.2
4,Indie,20.0
5,Pop,18.5


In [71]:
# Abbruchrate je Quelle 
abbruch_quelle = gesamt.groupby('quelle')['abgebrochen'].mean().mul(100).round(1).sort_values(ascending=False).reset_index()
abbruch_quelle.columns = ['Quelle', 'Abbruchrate %']
abbruch_quelle

,Quelle,Abbruchrate %
0,Autoplay,36.7
1,Playlist,19.0
2,Album,18.2
3,Suche,12.5


## Fazit Frage 4: Wo klicken Leute weg?

21,9% aller Streams werden innerhalb der ersten 30 Sekunden abgebrochen.

Die Abbruchrate unterscheidet sich deutlich nach Genre: Jazz (30,5%) und Electronic (29,3%) werden am häufigsten abgebrochen, Pop (18,5%) am seltensten.

Noch stärker unterscheidet sich die Abbruchrate nach der Quelle: Bei Autoplay (36,7%) wird fast doppelt so oft abgebrochen wie bei gezielter Suche (12,5%)   
– wer einen Song selbst aussucht, hört ihn deutlich häufiger zu Ende als wenn er automatisch weiterläuft.

## Gesamtfazit: Antwort an die Geschäftsführung

**Was wird gehört?** Die Top 5 Songs machen 24,7% aller Streams aus, Pop ist mit Abstand das beliebteste Genre.

**Von wem?** Die Altersgruppe 26-35 hört am meisten (44,2%). Premium- und Free-Nutzer haben einen ähnlichen Musikgeschmack, Premium-Nutzer streamen aber insgesamt mehr.

**Wann?** Die Streams verdoppeln sich fast über das Halbjahr (Januar bis April/Mai). Freitag und Samstag sind die stärksten Tage, morgens (7-9 Uhr) und abends (19-20 Uhr) wird am meisten gehört.

**Wo klicken Leute weg?** 21,9% aller Streams werden innerhalb der ersten 30 Sekunden abgebrochen. Autoplay-Songs werden deutlich häufiger abgebrochen (36,7%) als aktiv gesuchte (12,5%) – ein Hinweis, dass die automatische Wiedergabe verbessert werden könnte.

**Datenbasis:** 01.01.–30.06.2026, 5125 Streams nach Bereinigung, davon 4998 mit Katalog- und 4802 mit vollständigem Nutzerbezug.